In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [2]:
to_predict_mens = pd.read_csv("to_predict_mens.csv")

In [3]:
to_predict_mens[(to_predict_mens["GameRound"] == 1) & (to_predict_mens["Season"] == 2024)][["t1_TeamName", "t2_TeamName", "final_odds"]].head()

,t1_TeamName,t2_TeamName,final_odds
1318,Arizona,Long Beach St,-20.0
1319,Creighton,Akron,-12.5
1320,Dayton,Nevada,1.0
1321,Duquesne,BYU,7.5
1322,Gonzaga,McNeese St,-6.5


In [4]:
pd.set_option('display.max_columns', None)
1345
to_predict_mens[(to_predict_mens["Season"] == 2024)][["Team1", "t1_TeamName", "t1_adj_margin"]].drop_duplicates().sort_values(by="t1_adj_margin")

,Team1,t1_TeamName,t1_adj_margin
1317,1212,Grambling,-11.277963
2696,1224,Howard,-8.255369
1315,1447,Wagner,-8.062748
2719,1391,Stetson,-6.125959
2698,1286,Montana St,-4.311703
...,...,...,...
1324,1235,Iowa St,28.618729
2721,1388,St Mary's CA,28.632492
2730,1120,Auburn,28.847706
1338,1163,Connecticut,29.752956


In [5]:
to_predict_mens[(to_predict_mens["GameRound"] == 1) & (to_predict_mens["Season"] == 2024)
                & (to_predict_mens["Team2"] == 1345)]

,Unnamed: 0,type,ID,Pred,Season,Team1,Team2,Outcome,Gender,margin,t1_TeamName,t1_FirstD1Season,t1_LastD1Season,t2_TeamName,t2_FirstD1Season,t2_LastD1Season,final_odds,GameRound,t1_FGM,t1_FGA,t1_FGM3,t1_FGA3,t1_OR,t1_Ast,t1_TO,t1_Stl,t1_PF,t1_FTA,t1_FTM,t1_PointDiff,t2_FGM,t2_FGA,t2_FGM3,t2_FGA3,t2_OR,t2_Ast,t2_TO,t2_Stl,t2_PF,t2_FTA,t2_FTM,t2_PointDiff,t1_OrdinalRank,t2_OrdinalRank,t1_Seed,t2_Seed,seed_diff,t1_adj_oe,t1_adj_de,t1_adj_margin,t2_adj_oe,t2_adj_de,t2_adj_margin,t1_final_rank,t2_final_rank,t1_top8_TO_stdev,t1_top5_PRPG!_median,t1_top3_DR_median,t1_top5_STL_cv,t1_top3_Min%_median,t1_top8_TS_gini,t1_top3_USG_gini,t1_top8_BPM_weighted_mean,t2_top8_TO_stdev,t2_top5_PRPG!_median,t2_top3_DR_median,t2_top5_STL_cv,t2_top3_Min%_median,t2_top8_TS_gini,t2_top3_USG_gini,t2_top8_BPM_weighted_mean
2726,2728,Historical,2024_1212_1345,NaN,2024,1212,1345,0,M,-28,Grambling,1985,2025,Purdue,1985,2025,25.5,1,22.580645,52.354839,5.322581,15.612903,7.741935,9.16129,12.741935,7.129032,16.16129,20.741935,14.645161,-3.741935,28.515152,58.393939,8.333333,20.424242,11.030303,18.393939,10.969697,5.666667,14.363636,25.0,18.030303,13.242424,25.0,2.0,16,1,15,101.155352,112.433315,-11.277963,127.87684,100.690725,27.186115,66.780628,94.318459,5.996874,1.1,9.6,0.41779,66.6,0.050342,0.073737,-1.223557,3.197045,3.2,16.1,0.590937,77.4,0.10143,0.115556,7.184446


### Prepare data 

In [6]:
def prepare_data(to_predict_mens, season):

    to_predict_mens_first_round_train = to_predict_mens[(to_predict_mens["GameRound"] == 1)
                                                        & (to_predict_mens.final_odds.notnull())
                                                        & (to_predict_mens.Season < season)
                                                        ].copy()
    
    to_predict_mens_train = to_predict_mens[(to_predict_mens.Season < season)].copy()

    to_predict_mens_first_round_test = to_predict_mens[(to_predict_mens.Season == season)
                                                    & (to_predict_mens.GameRound == 1)
                                                    & (to_predict_mens.final_odds.notnull())
                                                    ].copy()

    to_predict_mens_other_rounds_test = to_predict_mens[(to_predict_mens.Season == season)
                                                    & (to_predict_mens.GameRound > 1)
                                                    ].copy()
    
    return to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test
    


### Odds Model Training

In [7]:
def train_odds_model(to_predict_mens_first_round_train):

    best_params = {"C": .1}
    model = LogisticRegression(**best_params)
    pipeline = make_pipeline(StandardScaler(), model)
    odds_model = pipeline.fit(to_predict_mens_first_round_train[["final_odds"]], to_predict_mens_first_round_train["Outcome"])
    
    return odds_model


### Statistics Model Training

In [8]:
def train_statistics_model(to_predict_mens_train, statistics_features):

    best_params = {"C": .1}
    model = LogisticRegression(**best_params)
    pipeline = make_pipeline(StandardScaler(), model)
    statistics_model = pipeline.fit(to_predict_mens_train[statistics_features], to_predict_mens_train["Outcome"])

    return statistics_model


### Inference

In [9]:
def inference(to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, odds_model, statistics_model, features):

    pred_proba = odds_model.predict_proba(to_predict_mens_first_round_test[["final_odds"]].copy())[:,1]
    to_predict_mens_first_round_test["odds_pred"] = pred_proba

    pred_proba = statistics_model.predict_proba(to_predict_mens_first_round_test[features].copy())[:,1]
    to_predict_mens_first_round_test["statistics_pred"] = pred_proba

    to_predict_mens_first_round_test["Pred"] = (to_predict_mens_first_round_test.odds_pred * .75) + \
                                                (to_predict_mens_first_round_test.statistics_pred * .25)
    

   

    pred_proba = statistics_model.predict_proba(to_predict_mens_other_rounds_test[features].copy())[:,1]
    to_predict_mens_other_rounds_test["Pred"] = pred_proba

    mens_sub_tmp = pd.concat([to_predict_mens_first_round_test,
            to_predict_mens_other_rounds_test], axis=0)
    
    mens_sub = mens_sub_tmp[["ID", "Pred"]]
    
    return mens_sub, mens_sub_tmp



### Full Pipeline

In [10]:
def run(to_predict_mens, season, statistics_features):
    
    # get data
    to_predict_mens_first_round_train, to_predict_mens_train, to_predict_mens_first_round_test, to_predict_mens_other_rounds_test= prepare_data(to_predict_mens, season)
    
    # train
    odds_model = train_odds_model(to_predict_mens_first_round_train)
    statistics_model = train_statistics_model(to_predict_mens_train, statistics_features)

    # inference
    mens_sub, mens_sub_tmp = inference(to_predict_mens_first_round_test, to_predict_mens_other_rounds_test, odds_model, statistics_model, statistics_features) 

    return mens_sub, mens_sub_tmp
    

In [11]:
statistics_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']
mens_sub, mens_sub_tmp = run(to_predict_mens, 2024, statistics_features)

### Womens

In [155]:
to_predict_women = pd.read_csv("to_predict_women.csv")

to_predict_women_train = to_predict_women[to_predict_women.Season != 2024] 
to_predict_women_test = to_predict_women[to_predict_women.Season == 2024] 


In [156]:
statistics_features = ['seed_diff', 't1_adj_margin', 't2_adj_margin']
best_params = {"C": .1}
model = LogisticRegression(**best_params)
pipeline = make_pipeline(StandardScaler(), model)
statistics_model = pipeline.fit(to_predict_women_train[statistics_features], to_predict_women_train["Outcome"])


In [157]:
pred_proba = statistics_model.predict_proba(to_predict_women_test[statistics_features].copy())[:,1]
to_predict_women_test["Pred"] = pred_proba
womens_sub = to_predict_women_test[["ID", "Pred"]]


/var/folders/2g/465yxy_x4g786jx2llr5xqh40000gn/T/ipykernel_13428/593629811.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_predict_women_test["Pred"] = pred_proba


In [159]:
final_sub = pd.concat([mens_sub, womens_sub], axis=0)

In [161]:
final_sub.to_csv("traditional_format_sub.csv")

### Sanity Check

In [30]:
mens_sub_tmp[mens_sub_tmp["t2_TeamName"] == "Connecticut"][["Pred", "Outcome", 't1_TeamName', 't2_TeamName',
                                                            "Team1", "Team2"]].tail(50)

,Pred,Outcome,t1_TeamName,t2_TeamName,Team1,Team2
2719,0.025109,0,Stetson,Connecticut,1391,1163
2741,0.117198,0,Northwestern,Connecticut,1321,1163
2749,0.165843,0,San Diego St,Connecticut,1361,1163
2756,0.251252,0,Illinois,Connecticut,1228,1163
2759,0.260613,0,Alabama,Connecticut,1104,1163
2761,0.436126,0,Purdue,Connecticut,1345,1163


In [28]:
mens_sub_tmp[mens_sub_tmp.Team1 == 1163][
    ["t1_adj_margin",'t1_final_rank', 't1_OrdinalRank']
    ].drop_duplicates()

,t1_adj_margin,t1_final_rank,t1_OrdinalRank
1338,29.752956,97.181957,5.0


In [17]:
mens_sub_tmp.shape

(126, 73)

In [47]:
mens_sub_tmp[mens_sub_tmp.t1_TeamName == "Duke"][["Pred", "Outcome", 't1_TeamName', 't2_TeamName', "GameRound"]].tail(15)

,Pred,Outcome,t1_TeamName,t2_TeamName,GameRound
1339,0.832048,1,Duke,Vermont,1
1361,0.800457,1,Duke,James Madison,2
1370,0.326038,1,Duke,Houston,3
2757,0.832355,0,Duke,NC State,4


In [13]:
brier_score_loss(mens_sub_tmp["Outcome"], mens_sub_tmp["Pred"])

0.18875757835489718

In [45]:
# Round 1 correctness
round1_correctness = mens_sub_tmp[mens_sub_tmp.GameRound == 1][["Pred", "Outcome", 't1_TeamName', 't2_TeamName', 
                                                                "t1_Seed", "t2_Seed","GameRound"]]
round1_correctness["Outcome_Pred"] = np.where(round1_correctness.Pred > .5, 1, 0 )
round1_correctness["Correct"] = np.where(round1_correctness["Outcome"] == round1_correctness["Outcome_Pred"], 1, 0)
round1_correctness["Correct"].mean()

0.625

In [46]:
round1_correctness[(
   (  (round1_correctness.t1_Seed >= 10) & (round1_correctness.Pred >= .20) )
)
].head(50)

,Pred,Outcome,t1_TeamName,t2_TeamName,t1_Seed,t2_Seed,GameRound,Outcome_Pred,Correct
1321,0.254401,1,Duquesne,BYU,11,6,1,0,0
1327,0.339417,1,NC State,Texas Tech,11,6,1,0,0
1330,0.515154,1,Oregon,South Carolina,11,6,1,1,1
1337,0.462046,1,Colorado,Florida,10,7,1,0,0
1340,0.328665,1,Grand Canyon,St Mary's CA,12,5,1,0,0
1342,0.361050,1,James Madison,Wisconsin,12,5,1,0,0
2701,0.521246,0,Nevada,Dayton,10,7,1,1,0
2703,0.278975,0,McNeese St,Gonzaga,12,5,1,0,1
2706,0.248915,0,Samford,Kansas,13,4,1,0,1
2713,0.412784,0,Colorado St,Texas,10,7,1,0,1


In [41]:
mens_sub_tmp[(mens_sub_tmp.t1_TeamName == "Wisconsin")][['statistics_pred', 'odds_pred', 't1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank', 'final_odds']]

,statistics_pred,odds_pred,t1_adj_margin,t2_adj_margin,t1_final_rank,t2_final_rank,t1_OrdinalRank,t2_OrdinalRank,final_odds
2723,0.656289,0.633242,20.598718,15.013969,87.404062,81.398593,25.0,24.0,-4.0


In [39]:
mens_sub_tmp.columns

Index(['Unnamed: 0', 'type', 'ID', 'Pred', 'Season', 'Team1', 'Team2',
       'Outcome', 'Gender', 'margin', 't1_TeamName', 't1_FirstD1Season',
       't1_LastD1Season', 't2_TeamName', 't2_FirstD1Season', 't2_LastD1Season',
       'final_odds', 'GameRound', 't1_FGM', 't1_FGA', 't1_FGM3', 't1_FGA3',
       't1_OR', 't1_Ast', 't1_TO', 't1_Stl', 't1_PF', 't1_FTA', 't1_FTM',
       't1_PointDiff', 't2_FGM', 't2_FGA', 't2_FGM3', 't2_FGA3', 't2_OR',
       't2_Ast', 't2_TO', 't2_Stl', 't2_PF', 't2_FTA', 't2_FTM',
       't2_PointDiff', 't1_OrdinalRank', 't2_OrdinalRank', 't1_Seed',
       't2_Seed', 'seed_diff', 't1_adj_oe', 't1_adj_de', 't1_adj_margin',
       't2_adj_oe', 't2_adj_de', 't2_adj_margin', 't1_final_rank',
       't2_final_rank', 't1_top8_TO_stdev', 't1_top5_PRPG!_median',
       't1_top3_DR_median', 't1_top5_STL_cv', 't1_top3_Min%_median',
       't1_top8_TS_gini', 't1_top3_USG_gini', 't1_top8_BPM_weighted_mean',
       't2_top8_TO_stdev', 't2_top5_PRPG!_median', 't2_top3_DR_m